In [ ]:
# Activacion de entorno

source mi_entorno/bin/activate

# Abrir Jupyter

jupyter notebook

In [14]:
# ================================
# 0. SETUP + LOGGING
# ================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
import numpy as np
import json
import time

log = []

def log_step(nombre, start, error=None):
    log.append({
        "paso": nombre,
        "tiempo_seg": round(time.time() - start, 2),
        "error": str(error) if error else ""
    })

# ================================
# 1. DRIVER
# ================================

start = time.time()

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

driver.get("https://resultadoelectoral.onpe.gob.pe/main/presidenciales")
time.sleep(5)

log_step("setup_driver", start)

# ================================
# 2. REGIONES
# ================================

departamentos = [
    ("010000","AMAZONAS"), ("020000","ANCASH"), ("030000","APURIMAC"),
    ("040000","AREQUIPA"), ("050000","AYACUCHO"), ("060000","CAJAMARCA"),
    ("240000","CALLAO"), ("070000","CUSCO"), ("080000","HUANCAVELICA"),
    ("090000","HUANUCO"), ("100000","ICA"), ("110000","JUNIN"),
    ("120000","LA LIBERTAD"), ("130000","LAMBAYEQUE"), ("140000","LIMA"),
    ("150000","LORETO"), ("160000","MADRE DE DIOS"), ("170000","MOQUEGUA"),
    ("180000","PASCO"), ("190000","PIURA"), ("200000","PUNO"),
    ("210000","SAN MARTIN"), ("220000","TACNA"), ("230000","TUMBES"),
    ("250000","UCAYALI")
]

# ================================
# 3. PROVINCIAS
# ================================

start = time.time()
provincias = []

for ubigeo_dep, nombre_dep in departamentos:
    try:
        url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/ubigeos/provincias?idEleccion=10&idAmbitoGeografico=1&idUbigeoDepartamento={ubigeo_dep}"
        driver.get(url)
        time.sleep(2)

        data = json.loads(driver.find_element("tag name", "body").text)["data"]

        for p in data:
            provincias.append({
                "departamento": nombre_dep,
                "ubigeo_dep": ubigeo_dep,
                "ubigeo_prov": p["ubigeo"],
                "provincia": p["nombre"]
            })

    except Exception as e:
        log_step(f"error_provincias_{nombre_dep}", start, e)

df_provincias = pd.DataFrame(provincias)
df_provincias.to_csv("provincias.csv", index=False)

log_step("provincias", start)

# ================================
# 4. VOTOS
# ================================

start = time.time()
votos = []

for _, row in df_provincias.iterrows():
    try:
        url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/eleccion-presidencial/participantes-ubicacion-geografica-nombre?tipoFiltro=ubigeo_nivel_02&idAmbitoGeografico=1&ubigeoNivel1={row['ubigeo_dep']}&ubigeoNivel2={row['ubigeo_prov']}&idEleccion=10"

        driver.get(url)
        time.sleep(2)

        data = json.loads(driver.find_element("tag name", "body").text)["data"]

        for fila in data:
            votos.append({
                "departamento": row["departamento"],
                "provincia": row["provincia"],
                "candidato": fila["nombreCandidato"],
                "partido": fila["nombreAgrupacionPolitica"],
                "votos": fila.get("totalVotosValidos", 0)
            })

    except Exception as e:
        log_step(f"error_votos_{row['provincia']}", start, e)

df_votos = pd.DataFrame(votos)
df_votos.to_csv("votos.csv", index=False)

log_step("votos", start)

# ================================
# 5. ACTAS
# ================================

start = time.time()
actas = []

for _, row in df_provincias.iterrows():
    try:
        url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/resumen-general/totales?idAmbitoGeografico=1&idEleccion=10&tipoFiltro=ubigeo_nivel_02&idUbigeoDepartamento={row['ubigeo_dep']}&idUbigeoProvincia={row['ubigeo_prov']}"

        driver.get(url)
        time.sleep(2)

        data = json.loads(driver.find_element("tag name", "body").text)["data"]

        actas.append({
            "departamento": row["departamento"],
            "provincia": row["provincia"],
            "avance_pct": data["actasContabilizadas"]  # 🔥 correcto
        })

    except Exception as e:
        log_step(f"error_actas_{row['provincia']}", start, e)

df_actas = pd.DataFrame(actas)
df_actas.to_csv("actas.csv", index=False)

log_step("actas", start)

# ================================
# 6. MERGE
# ================================

start = time.time()

df_final = df_votos.merge(df_actas, on=["departamento","provincia"])
df_final.to_csv("resultado_provincial_completo.csv", index=False)

log_step("merge", start)

# ================================
# 7. EXTRANJERO
# ================================

start = time.time()

extranjero = [
    ("910000","AFRICA"), ("920000","AMERICA"),
    ("930000","ASIA"), ("940000","EUROPA"), ("950000","OCEANIA")
]

votos_ext = []
actas_ext = []

for ubigeo, nombre in extranjero:
    try:
        # votos
        url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/eleccion-presidencial/participantes-ubicacion-geografica-nombre?tipoFiltro=ubigeo_nivel_01&idAmbitoGeografico=2&ubigeoNivel1={ubigeo}&idEleccion=10"
        driver.get(url)
        time.sleep(2)

        data_v = json.loads(driver.find_element("tag name", "body").text)["data"]

        for fila in data_v:
            votos_ext.append({
                "departamento": f"EXTERIOR_{nombre}",
                "provincia": f"EXTERIOR_{nombre}",
                "candidato": fila["nombreCandidato"],
                "partido": fila["nombreAgrupacionPolitica"],
                "votos": fila.get("totalVotosValidos", 0)
            })

        # actas
        url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/resumen-general/totales?idAmbitoGeografico=2&idEleccion=10&tipoFiltro=ubigeo_nivel_01&idUbigeoDepartamento={ubigeo}"
        driver.get(url)
        time.sleep(2)

        data_a = json.loads(driver.find_element("tag name", "body").text)["data"]

        actas_ext.append({
            "departamento": f"EXTERIOR_{nombre}",
            "provincia": f"EXTERIOR_{nombre}",
            "avance_pct": data_a["actasContabilizadas"]
        })

    except Exception as e:
        log_step(f"error_exterior_{nombre}", start, e)

df_ext = pd.DataFrame(votos_ext).merge(pd.DataFrame(actas_ext), on=["departamento","provincia"])

df_total = pd.concat([df_final, df_ext], ignore_index=True)

log_step("extranjero", start)

# ================================
# 8. EXTRAPOLACIÓN
# ================================

start = time.time()

df_total["avance_pct"] = df_total["avance_pct"].replace(0, np.nan)

df_total["votos_estimados"] = df_total["votos"] / (df_total["avance_pct"] / 100)

df_total["votos_restantes"] = df_total["votos_estimados"] - df_total["votos"]

df_total.to_csv("base_intermedia_proyeccion.csv", index=False)

log_step("extrapolacion", start)

# ================================
# 9. MÉTRICAS
# ================================

start = time.time()

totales = df_total.groupby("departamento")["votos_estimados"].sum().reset_index()
totales = totales.rename(columns={"votos_estimados": "total_region"})

df_total = df_total.merge(totales, on="departamento")

df_total["pct_region_actual"] = df_total["votos"] / df_total["total_region"] * 100
df_total["pct_region_proyectado"] = df_total["votos_estimados"] / df_total["total_region"] * 100

df_total.to_csv("proyeccion_provincial_completa.csv", index=False)

log_step("metricas", start)

# ================================
# 10. NACIONAL
# ================================

start = time.time()

df_nacional = df_total.groupby(["candidato","partido"], as_index=False)["votos_estimados"].sum()
df_nacional = df_nacional.sort_values(by="votos_estimados", ascending=False)

df_nacional.to_csv("nacional_resumen.csv", index=False)

log_step("nacional", start)

# ================================
# 11. LOG FINAL
# ================================

df_log = pd.DataFrame(log)
df_log.to_csv("log_proceso.csv", index=False)

print("\n✅ TODO TERMINADO")
print("Archivos generados:")
print("- resultado_provincial_completo.csv")
print("- base_intermedia_proyeccion.csv")
print("- proyeccion_provincial_completa.csv")
print("- nacional_resumen.csv")
print("- log_proceso.csv")


✅ TODO TERMINADO
Archivos generados:
- resultado_provincial_completo.csv
- base_intermedia_proyeccion.csv
- proyeccion_provincial_completa.csv
- nacional_resumen.csv
- log_proceso.csv


In [4]:
df["avance_pct"].describe()

count    7296.000000
mean       69.131328
std        28.645066
min         0.734000
25%        55.653000
50%        80.358500
75%        92.628000
max       100.000000
Name: avance_pct, dtype: float64

In [5]:
df[df["avance_pct"] < 30].groupby("candidato")["votos"].sum().sort_values(ascending=False)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO            33613
                                            31721
KEIKO SOFIA FUJIMORI HIGUCHI                16475
RICARDO PABLO BELMONT CASSINELLI            11481
PABLO ALFONSO LOPEZ CHAU NAVA                5403
CARLOS GONSALO ALVAREZ LOAYZA                4326
JORGE NIETO MONTESINOS                       4017
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA         3536
JOSE LEON LUNA GALVEZ                        2664
RONALD DARWIN ATENCIO SOTOMAYOR              2236
ALFONSO CARLOS ESPA Y GARCES-ALVEAR          1746
MARIA SOLEDAD PEREZ TELLO                    1485
LUIS FERNANDO OLIVERA VEGA                   1472
YONHY LESCANO ANCIETA                        1372
CESAR ACUÑA PERALTA                          1298
GEORGE PATRICK FORSYTH SOMMER                1283
HERBERT CALLER GUTIERREZ                     1088
MARIO ENRIQUE VIZCARRA CORNEJO               1048
CHARLIE CARRASCO SALAZAR                      992
VLADIMIR ROY CERRON ROJAS               

In [6]:
df_bajo = df[df["avance_pct"] < 30]

df_bajo.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO    33613
                                    31721
KEIKO SOFIA FUJIMORI HIGUCHI        16475
RICARDO PABLO BELMONT CASSINELLI    11481
PABLO ALFONSO LOPEZ CHAU NAVA        5403
Name: votos, dtype: int64

In [7]:
df_alto = df[df["avance_pct"] > 80]

df_alto.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
                                        1845903
KEIKO SOFIA FUJIMORI HIGUCHI            1789646
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA    1565954
JORGE NIETO MONTESINOS                  1438624
RICARDO PABLO BELMONT CASSINELLI        1119108
Name: votos, dtype: int64

In [8]:
import numpy as np
import pandas as pd
import time

# ================================
# 1. BASE LIMPIA (INPUT)
# ================================

df_mc_base = df_final.copy()

# evitar división por cero
df_mc_base["avance_pct"] = df_mc_base["avance_pct"].replace(0, np.nan)

# extrapolación determinística base
df_mc_base["votos_estimados"] = (
    df_mc_base["votos"] / (df_mc_base["avance_pct"] / 100)
)

# residual (lo que falta por distribuir)
df_mc_base["votos_restantes"] = (
    df_mc_base["votos_estimados"] - df_mc_base["votos"]
)

# limpieza de seguridad
df_mc_base = df_mc_base.dropna(subset=["votos_restantes"])
df_mc_base["votos_restantes"] = df_mc_base["votos_restantes"].clip(lower=0)

# ================================
# 2. PARÁMETROS MONTE CARLO
# ================================

N = 5000
step_print = 500
start_time = time.time()

resultados = []

# nivel de agregación territorial (AJUSTA AQUÍ)
group_cols = ["provincia"]   # o ["departamento"] o ["distrito"]

# nombres exactos candidatos
RLA = "RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA"
SANCH = "ROBERTO HELBERT SANCHEZ PALOMINO"
NIETO = "JORGE NIETO MONTESINOS"

# ================================
# 3. MONTE CARLO
# ================================

for i in range(N):

    sim = df_mc_base.copy()

    # ----------------------------
    # shock aleatorio territorial
    # ----------------------------
    ruido = np.random.lognormal(mean=0, sigma=0.12, size=len(sim))
    sim["peso_sim"] = ruido

    # normalizar dentro de grupo territorial
    sim["peso_sim"] = sim.groupby(group_cols)["peso_sim"].transform(
        lambda x: x / x.sum()
    )

    # ----------------------------
    # redistribución de votos restantes
    # ----------------------------
    sim["votos_simulados"] = (
        sim["votos"] + sim["peso_sim"] * sim["votos_restantes"]
    )

    # ----------------------------
    # agregación nacional
    # ----------------------------
    nacional = sim.groupby("candidato")["votos_simulados"].sum()

    votos_rla = nacional.get(RLA, 0)
    votos_sanchez = nacional.get(SANCH, 0)
    votos_nieto = nacional.get(NIETO, 0)

    resultados.append({
        "rla": votos_rla,
        "sanchez": votos_sanchez,
        "nieto": votos_nieto,
        "rla_gana": votos_rla > votos_sanchez
    })

    # ----------------------------
    # progreso
    # ----------------------------
    if (i + 1) % step_print == 0:
        elapsed = time.time() - start_time
        eta = (elapsed / (i + 1)) * (N - i - 1)

        print(
            f"🔄 {i+1}/{N} | "
            f"{(i+1)/N:.1%} | "
            f"ETA: {eta:.1f}s"
        )

# ================================
# 4. RESULTADOS FINALES
# ================================

df_res = pd.DataFrame(resultados)

df_res["diff"] = df_res["rla"] - df_res["sanchez"]

print("\n🔥 RESULTADOS MONTE CARLO\n")

print("Probabilidad RLA gana:",
      round(df_res["rla_gana"].mean() * 100, 2), "%")

print("Probabilidad Sánchez gana:",
      round((1 - df_res["rla_gana"].mean()) * 100, 2), "%")

print("\n📊 Distribución diferencia (RLA - Sánchez):")

print(df_res["diff"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]))

🔄 500/5000 | 10.0% | ETA: 120.5s
🔄 1000/5000 | 20.0% | ETA: 108.0s
🔄 1500/5000 | 30.0% | ETA: 95.8s
🔄 2000/5000 | 40.0% | ETA: 83.5s
🔄 2500/5000 | 50.0% | ETA: 69.2s
🔄 3000/5000 | 60.0% | ETA: 55.7s
🔄 3500/5000 | 70.0% | ETA: 41.7s
🔄 4000/5000 | 80.0% | ETA: 27.8s
🔄 4500/5000 | 90.0% | ETA: 13.9s
🔄 5000/5000 | 100.0% | ETA: 0.0s

🔥 RESULTADOS MONTE CARLO

Probabilidad RLA gana: 0.0 %
Probabilidad Sánchez gana: 100.0 %

📊 Distribución diferencia (RLA - Sánchez):
0.05   -61386.158209
0.25   -61145.889474
0.50   -60969.891891
0.75   -60781.161285
0.95   -60492.315825
Name: diff, dtype: float64


In [12]:
import pandas as pd

df_res = pd.DataFrame(resultados)

# Probabilidades
prob_rla = df_res["gana_rla"].mean()
prob_sanchez = 1 - prob_rla

# Diferencia promedio
df_res["diff"] = df_res["rla"] - df_res["sanchez"]
diff_media = df_res["diff"].mean()

print("\n🔥 RESULTADOS MONTE CARLO\n")
print(f"Probabilidad RLA gana: {prob_rla:.2%}")
print(f"Probabilidad Sánchez gana: {prob_sanchez:.2%}")
print(f"Diferencia promedio (RLA - Sánchez): {diff_media:,.0f} votos")

# Intervalos
print("\n📊 Distribución de diferencia (RLA - Sánchez):")
print(df_res["diff"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]))


🔥 RESULTADOS MONTE CARLO

Probabilidad RLA gana: 100.00%
Probabilidad Sánchez gana: 0.00%
Diferencia promedio (RLA - Sánchez): 178,549 votos

📊 Distribución de diferencia (RLA - Sánchez):
0.05    172890.242109
0.25    176312.255743
0.50    178474.430879
0.75    180806.148868
0.95    184019.540674
Name: diff, dtype: float64
